# OCR Evaluation · Полный пайплайн

Объединённый ноутбук для Google Colab. Содержит все этапы:
1. **Setup & Dataset** — установка зависимостей, загрузка OmniDocBench, сохранение subset
2. **DeepSeek-OCR** — инференс модели `deepseek-ai/DeepSeek-OCR`
3. **mPLUG-DocOwl 2** — инференс модели `mPLUG/DocOwl2`
4. **olmOCR** — инференс модели `allenai/olmOCR-7B-0225-preview`
5. **MonkeyOCR** — инференс модели `echo840/MonkeyOCR`

> **Совет:** В Colab используйте GPU runtime (T4/L4/A100). Запускайте секции последовательно — каждая модель освобождает GPU перед загрузкой следующей.

---
# Часть 1 · Подготовка среды и загрузка OmniDocBench

Этот раздел готовит среду к экспериментам:
1. Устанавливает зависимости из `requirements.txt`.
2. Скачивает датасет **OmniDocBench v1.6** (`opendatalab/OmniDocBench` на HuggingFace).
3. Загружает разметку, фильтрует страницы типа `academic_literature` (научные статьи arXiv-стиля).
4. Показывает превью первой страницы и её ground-truth.
5. Сохраняет subset в `data/subset.json` — все модели используют один и тот же список страниц.

In [ ]:
# В Colab можно использовать GPU runtime → T4/L4/A100
!nvidia-smi || echo 'GPU не подключена'

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
# === Colab / Kaggle bootstrap =================================================
# В Colab клонируем репозиторий проекта (предполагается, что код выложен
# в GitHub) и переходим в его корень. Для локального запуска просто
# проверьте, что текущая рабочая директория — корень ocr_eval/.
import os, sys, subprocess, pathlib

REPO_NAME = "ocr_eval"
if not pathlib.Path(REPO_NAME).exists():
    # Замените URL на ваш форк, если работаете в Colab
    # !git clone https://github.com/<your-username>/ocr_eval.git
    pass

if pathlib.Path(REPO_NAME).exists():
    os.chdir(REPO_NAME)
sys.path.insert(0, str(pathlib.Path.cwd() / "src"))
print("CWD =", os.getcwd())

In [1]:
from src.dataset_loader import download_omnidocbench, load_omnidocbench, iter_pages

DATA_ROOT = 'data/OmniDocBench'
download_omnidocbench(DATA_ROOT, source='huggingface')
print('OK, датасет в', DATA_ROOT)

ModuleNotFoundError: No module named 'PIL'

In [ ]:
# Берём 100 случайных страниц-научных публикаций для экспериментов
items = load_omnidocbench(
    root=DATA_ROOT,
    page_types=['academic_literature'],
    languages=['english'],
    subset_size=100,
    seed=42,
)
print(f'Отобрано страниц: {len(items)}')
items[0].to_dict() if items else 'empty'

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

first = items[0]
img = Image.open(Path(DATA_ROOT) / first.image_path).convert('RGB')
fig, ax = plt.subplots(figsize=(8, 11))
ax.imshow(img); ax.axis('off')
ax.set_title(f'{first.page_id}  ·  {first.page_type}')
plt.show()

In [ ]:
# Сохраняем список выбранных страниц.
# Все 4 блока инференса используют один и тот же subset — так результаты сравнимы.
import json, pathlib
subset_path = pathlib.Path('data/subset.json')
subset_path.parent.mkdir(parents=True, exist_ok=True)
subset_path.write_text(
    json.dumps([x.to_dict() for x in items], ensure_ascii=False),
    encoding='utf-8',
)
print('сохранено:', subset_path, subset_path.stat().st_size, 'байт')

---
# Часть 2 · DeepSeek-OCR

**DeepSeek-OCR** — открытая мульти-модальная модель (≈3B параметров) от DeepSeek-AI; поддерживает grounding-промпт и markdown-вывод. Чекпоинт: [`deepseek-ai/DeepSeek-OCR`](https://huggingface.co/deepseek-ai/DeepSeek-OCR).

In [ ]:
from src.utils import load_config, JsonlWriter, Timer, cuda_free, gpu_info, already_processed_ids
from src.io_records import PredictionRecord

cfg = load_config('configs/deepseek_ocr.yaml')
print(gpu_info())
cfg

In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer

MODEL_REPO = cfg['model']['hf_repo']
tokenizer = AutoTokenizer.from_pretrained(MODEL_REPO, trust_remote_code=True)
model = AutoModel.from_pretrained(
    MODEL_REPO,
    trust_remote_code=True,
    torch_dtype=getattr(torch, cfg['model']['torch_dtype']),
    device_map=cfg['model']['device_map'],
).eval()
print(gpu_info())

In [ ]:
import json, pathlib
from src.dataset_loader import GroundTruth

DATA_ROOT = pathlib.Path('data/OmniDocBench')
subset = [GroundTruth(**rec) for rec in json.loads(
    pathlib.Path('data/subset.json').read_text(encoding='utf-8'))]
print(f'subset: {len(subset)} страниц')

In [ ]:
import os, time, traceback
from pathlib import Path
from PIL import Image

out_dir = Path(cfg['output']['results_dir'])
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / 'predictions.jsonl'
done = already_processed_ids(out_path)
print(f'уже обработано: {len(done)} / {len(subset)}')

with JsonlWriter(out_path) as w:
    for gt in subset:
        if gt.page_id in done:
            continue
        img_path = DATA_ROOT / gt.image_path
        if not img_path.exists():
            continue
        rec = PredictionRecord(page_id=gt.page_id, model='deepseek_ocr')
        try:
            with Timer('infer') as t:
                # API DeepSeek-OCR (см. README модели):
                # model.infer(tokenizer, prompt, image_file, output_path,
                #             base_size, image_size, crop_mode, save_results, test_compress)
                tmp_out = out_dir / f'{gt.page_id}'
                tmp_out.mkdir(parents=True, exist_ok=True)
                result_md = model.infer(
                    tokenizer,
                    prompt=cfg['inference']['prompt'],
                    image_file=str(img_path),
                    output_path=str(tmp_out),
                    base_size=cfg['inference']['base_size'],
                    image_size=cfg['inference']['image_size'],
                    crop_mode=cfg['inference']['crop_mode'],
                    save_results=False,
                    test_compress=cfg['inference']['test_compress'],
                )
            rec.full_text = result_md if isinstance(result_md, str) else str(result_md)
            rec.raw_output = rec.full_text
            rec.inference_time_s = t.elapsed
        except Exception as e:
            rec.error = f'{type(e).__name__}: {e}'
            traceback.print_exc()
        w.write(rec.to_dict())
print('готово →', out_path)

In [ ]:
# Превью одного результата
from src.utils import read_jsonl
preds = read_jsonl(out_path)
print(len(preds), 'записей')
print(preds[0]['full_text'][:1000])

In [ ]:
# Освобождаем GPU перед следующей моделью
del model; cuda_free(); print(gpu_info())

---
# Часть 3 · mPLUG-DocOwl 2

**mPLUG-DocOwl 2** — модель Alibaba для понимания документов с shape-adaptive cropping. Чекпоинт: [`mPLUG/DocOwl2`](https://huggingface.co/mPLUG/DocOwl2).

In [ ]:
from src.utils import load_config, JsonlWriter, Timer, cuda_free, gpu_info, already_processed_ids
from src.io_records import PredictionRecord

cfg = load_config('configs/mplug_docowl.yaml')
cfg

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_REPO = cfg['model']['hf_repo']
tokenizer = AutoTokenizer.from_pretrained(MODEL_REPO, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_REPO,
    trust_remote_code=True,
    torch_dtype=getattr(torch, cfg['model']['torch_dtype']),
    device_map=cfg['model']['device_map'],
).eval()
print(gpu_info())

In [ ]:
import json, pathlib
from src.dataset_loader import GroundTruth

DATA_ROOT = pathlib.Path('data/OmniDocBench')
subset = [GroundTruth(**rec) for rec in json.loads(
    pathlib.Path('data/subset.json').read_text(encoding='utf-8'))]
print(f'subset: {len(subset)} страниц')

In [ ]:
# DocOwl ожидает PIL.Image + текстовый промпт; в зависимости от версии
# либо есть метод model.chat(images=[img], query=prompt), либо нужно
# собрать messages вручную. Универсальный путь — через preprocessor:
from transformers import AutoProcessor
try:
    processor = AutoProcessor.from_pretrained(MODEL_REPO, trust_remote_code=True)
except Exception:
    processor = None
    print('processor не нужен — используем model.chat() напрямую')

In [ ]:
import traceback
from pathlib import Path
from PIL import Image

out_path = Path(cfg['output']['results_dir']) / 'predictions.jsonl'
out_path.parent.mkdir(parents=True, exist_ok=True)
done = already_processed_ids(out_path)

PROMPT = cfg['inference']['prompt']
MAX_NEW = cfg['inference']['max_new_tokens']

with JsonlWriter(out_path) as w:
    for gt in subset:
        if gt.page_id in done: continue
        img_path = DATA_ROOT / gt.image_path
        if not img_path.exists(): continue
        rec = PredictionRecord(page_id=gt.page_id, model='mplug_docowl')
        try:
            img = Image.open(img_path).convert('RGB')
            with Timer('infer') as t:
                if hasattr(model, 'chat'):
                    out = model.chat(image=img, msgs=[{'role':'user','content': PROMPT}],
                                     tokenizer=tokenizer, sampling=False, max_new_tokens=MAX_NEW)
                else:
                    inputs = processor(images=img, text=PROMPT, return_tensors='pt').to(model.device)
                    with torch.no_grad():
                        gen = model.generate(**inputs, max_new_tokens=MAX_NEW, do_sample=False)
                    out = processor.batch_decode(gen, skip_special_tokens=True)[0]
            rec.full_text = out if isinstance(out, str) else str(out)
            rec.raw_output = rec.full_text
            rec.inference_time_s = t.elapsed
        except Exception as e:
            rec.error = f'{type(e).__name__}: {e}'
            traceback.print_exc()
        w.write(rec.to_dict())
print('готово →', out_path)

In [ ]:
# Освобождаем GPU перед следующей моделью
del model; cuda_free(); print(gpu_info())

---
# Часть 4 · olmOCR (AllenAI)

**olmOCR** — пайплайн от AllenAI поверх Qwen2-VL-7B-Instruct, дообученный на ~250K страницах. Чекпоинт: [`allenai/olmOCR-7B-0225-preview`](https://huggingface.co/allenai/olmOCR-7B-0225-preview).

> Для OmniDocBench используется упрощённый путь без anchor-текста (в отчёте отметить, что это даёт небольшой проигрыш по сравнению с полным пайплайном).

In [ ]:
# Системные зависимости: poppler нужен для anchor-текста
!apt-get -qq install -y poppler-utils ttf-mscorefonts-installer 2>&1 | tail -1
!pip install -q olmocr

In [ ]:
from src.utils import load_config, JsonlWriter, Timer, cuda_free, gpu_info, already_processed_ids
from src.io_records import PredictionRecord
cfg = load_config('configs/olmocr.yaml')
cfg

In [ ]:
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor

MODEL_REPO = cfg['model']['hf_repo']
processor = AutoProcessor.from_pretrained(MODEL_REPO)
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_REPO,
    torch_dtype=getattr(torch, cfg['model']['torch_dtype']),
    device_map=cfg['model']['device_map'],
).eval()
print(gpu_info())

In [ ]:
import json, pathlib
from src.dataset_loader import GroundTruth

DATA_ROOT = pathlib.Path('data/OmniDocBench')
subset = [GroundTruth(**rec) for rec in json.loads(
    pathlib.Path('data/subset.json').read_text(encoding='utf-8'))]
print(f'subset: {len(subset)} страниц')

In [ ]:
import traceback, base64, io
from pathlib import Path
from PIL import Image

out_path = Path(cfg['output']['results_dir']) / 'predictions.jsonl'
out_path.parent.mkdir(parents=True, exist_ok=True)
done = already_processed_ids(out_path)

MAX_NEW = cfg['inference']['max_new_tokens']
TEMP    = cfg['inference']['temperature']

OLMOCR_PROMPT = (
    'Below is the image of one page of a document. Just return the plain text '
    'representation of this document as if you were reading it naturally. '
    'Convert equations to LaTeX and tables to HTML. Do not hallucinate.'
)

with JsonlWriter(out_path) as w:
    for gt in subset:
        if gt.page_id in done: continue
        img_path = DATA_ROOT / gt.image_path
        if not img_path.exists(): continue
        rec = PredictionRecord(page_id=gt.page_id, model='olmocr')
        try:
            img = Image.open(img_path).convert('RGB')
            messages = [{
                'role': 'user',
                'content': [
                    {'type': 'image'},
                    {'type': 'text', 'text': OLMOCR_PROMPT},
                ],
            }]
            text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            inputs = processor(text=[text], images=[img], padding=True, return_tensors='pt').to(model.device)
            with Timer('infer') as t, torch.no_grad():
                gen = model.generate(**inputs, max_new_tokens=MAX_NEW, do_sample=TEMP > 0,
                                     temperature=TEMP if TEMP > 0 else 1.0)
            out = processor.batch_decode(
                gen[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0]
            rec.full_text = out
            rec.raw_output = out
            rec.inference_time_s = t.elapsed
        except Exception as e:
            rec.error = f'{type(e).__name__}: {e}'
            traceback.print_exc()
        w.write(rec.to_dict())
print('готово →', out_path)

In [ ]:
# Освобождаем GPU перед следующей моделью
del model; cuda_free(); print(gpu_info())

---
# Часть 5 · MonkeyOCR

**MonkeyOCR** — модель с парадигмой **Structure → Recognition → Relation**. Чекпоинт: [`echo840/MonkeyOCR`](https://huggingface.co/echo840/MonkeyOCR).

In [ ]:
from src.utils import load_config, JsonlWriter, Timer, cuda_free, gpu_info, already_processed_ids
from src.io_records import PredictionRecord
cfg = load_config('configs/monkeyocr.yaml')
cfg

In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer
MODEL_REPO = cfg['model']['hf_repo']
tokenizer = AutoTokenizer.from_pretrained(MODEL_REPO, trust_remote_code=True)
model = AutoModel.from_pretrained(
    MODEL_REPO,
    trust_remote_code=True,
    torch_dtype=getattr(torch, cfg['model']['torch_dtype']),
    device_map=cfg['model']['device_map'],
).eval()
print(gpu_info())

In [ ]:
import json, pathlib
from src.dataset_loader import GroundTruth

DATA_ROOT = pathlib.Path('data/OmniDocBench')
subset = [GroundTruth(**rec) for rec in json.loads(
    pathlib.Path('data/subset.json').read_text(encoding='utf-8'))]
print(f'subset: {len(subset)} страниц')

In [ ]:
# MonkeyOCR имеет встроенный метод model.chat_full_page(...) (см. README модели).
# Если интерфейс изменится — заменить на актуальный.
import traceback
from pathlib import Path
from PIL import Image

out_path = Path(cfg['output']['results_dir']) / 'predictions.jsonl'
out_path.parent.mkdir(parents=True, exist_ok=True)
done = already_processed_ids(out_path)

with JsonlWriter(out_path) as w:
    for gt in subset:
        if gt.page_id in done: continue
        img_path = DATA_ROOT / gt.image_path
        if not img_path.exists(): continue
        rec = PredictionRecord(page_id=gt.page_id, model='monkeyocr')
        try:
            img = Image.open(img_path).convert('RGB')
            with Timer('infer') as t, torch.no_grad():
                if hasattr(model, 'chat_full_page'):
                    out = model.chat_full_page(tokenizer, img,
                                               max_new_tokens=cfg['inference']['max_new_tokens'])
                else:
                    out = model.chat(tokenizer, img,
                                     query='Convert this page to markdown',
                                     max_new_tokens=cfg['inference']['max_new_tokens'])
            rec.full_text = out if isinstance(out, str) else str(out)
            rec.raw_output = rec.full_text
            rec.inference_time_s = t.elapsed
        except Exception as e:
            rec.error = f'{type(e).__name__}: {e}'
            traceback.print_exc()
        w.write(rec.to_dict())
print('готово →', out_path)

In [ ]:
# Освобождаем GPU
del model; cuda_free(); print(gpu_info())